# LABORATORIO 3: Firmas digitales 

## Ejercicio 1: Análisis y Verificación de Firmas Digitales

### Razonamiento previo


- Firma Digital vs Cifrado: El cifrado protege la confidencialidad (solo el destinatario con la clave privada puede leer el mensaje). La firma digital protege la autenticidad, la integridad y el no repudio (garantiza que el mensaje fue enviado por el dueño de la clave privada y que no ha sido alterado).

- El proceso de firmar: El emisor calcula una función hash del mensaje (una huella digital única) y cifra ese hash utilizando su clave privada.

- El proceso de verificar: El receptor descifra la firma utilizando la clave pública del emisor para recuperar el hash original, y lo compara con el hash calculado localmente sobre el mensaje recibido. Si ambos hashes coinciden exactamente, la firma es válida.

version original:

In [1]:
from cryptography.hazmat.primitives import hashes
from cryptography.hazmat.primitives.asymmetric import rsa, padding

# Generación de claves
private_key = rsa.generate_private_key(
 public_exponent=65537,
 key_size=2048
)
public_key = private_key.public_key()

# Mensaje original
message = b"Transfer 100 euros"

# FIRMA
signature = private_key.sign(
 message,
 padding.PKCS1v15(),
 hashes.SHA256()
)

# VERIFICACIÓN
try:
 public_key.verify(
 signature,
 message + b" extra",
  padding.PKCS1v15(),
 hashes.SHA1()
 )
 print("Firma válida")
except Exception:
 print("Firma inválida")

Firma inválida


version corregida:

In [2]:
from cryptography.hazmat.primitives import hashes
from cryptography.hazmat.primitives.asymmetric import rsa, padding

# 1. Generación de claves RSA (2048 bits es el estándar seguro actual)
private_key = rsa.generate_private_key(
    public_exponent=65537,
    key_size=2048
)
public_key = private_key.public_key()

# 2. Mensaje original
message = b"Transfer 100 euros"

# 3. FIRMA DIGITAL: Se usa la clave privada, padding PKCS1v15 y hash SHA256
signature = private_key.sign(
    message,
    padding.PKCS1v15(),
    hashes.SHA256()
)

print("¡Claves generadas y mensaje firmado con éxito!")
print(
    f"Mensaje original: {message.decode()}\n"
    f"Firma digital (en bytes): {signature}\n"
    f"Firma digital (en hexadecimal): {signature.hex()}"
)

¡Claves generadas y mensaje firmado con éxito!
Mensaje original: Transfer 100 euros
Firma digital (en bytes): b'\x05\xa5\xad\xc1\xb9\x17\x90\xc2H\x0b\x82X\x99\xdc\xdc\xdf\x00\x95\x966\xe2j\xee\xf8\x96\xbb\xe4\xce\x9e\xdb\x10\xca\xb5\x9ep\xb2P\xd3\xcf$\xa1Q\xa5\xdd2\x92\xb5\x9f_~\xc6\x96\xe8T\x15Qf\xc0\xd3%\xbf\xfbp\xa0\xe2 \'\xb6>\x17\x19\xd0\xe7\x9f\x1fX\x13\x1b\x0f\x1e%#;\x9f\xfe\x11\x91e\xdd?\x81I\x91i\xae\x92\xc6\x99\xa4 \xd0\x80"w\x0f\xd2K\xddy\xc8K$\x81$\xb5!\xb1\xdb\x84\x15\xca\xd2I\x88\xb9\x87\x16)l\xf7\xf1\x06\x9f\xdae=\xea\xde\xf7\xf7\x9b\xed\xad\xd2#\xaf\xa2\x96/\xe72\xda.\xb0G\x07\n\xeai\x19\xaa\\\x01\x9f\xbdq\xca\xcc\xe5t\x13\x13qZK\xf2t\x93\xb0\x90\xd7Q\xf6E*\xc4\xe0\xc6S\x9a\xa5\x06#A\xf5lV\xec\xba\xea>UW\xda\xf7\xaf\xc5\x96\xa0\x86\x85\xbd\x1c\xa6$\x90B\xd5\xff\x1f\t\xc4jc\xb1\xa9\x17N~\xd9\xb3X)\xccWl=\x90\xd6p\x03\xc9\x81 (\x82\xa1\xee\x85\xf2\xd3\x84l2_\xd9'
Firma digital (en hexadecimal): 05a5adc1b91790c2480b825899dcdcdf00959636e26aeef896bbe4ce9edb10cab59e70b250d3cf

a) ¿Qué hace este código en términos generales?

- Generación de claves: Crea un par de claves asimétricas RSA (una privada para firmar y otra pública para verificar).

- Firma: Toma el mensaje "Transfer 100 euros", le aplica una función hash (SHA256) y un esquema de relleno (PKCS1v15), y cifra el resultado con la clave privada para generar la signature.

- Verificación (con errores en el planteamiento original): Intenta comprobar la autenticidad de la firma utilizando la clave pública, pero introduce cambios en el proceso de validación (message + b" extra" y hashes.SHA1()).

b) ¿Qué problema hay en la verificación del mensaje? ¿Qué ataque podría ocurrir?

- El problema: Hay dos discrepancias críticas entre la firma y la verificación en el código original: Se firmó el mensaje message, pero en la verificación se evalúa message + b" extra" (el mensaje ha sido alterado/modificado en la entrada). Se firmó utilizando la función hash hashes.SHA256(), pero se intenta verificar utilizando hashes.SHA1().

- El ataque potencial: Si una aplicación valida firmas de manera laxa o permite discrepancias en los parámetros criptográficos (como aceptar hashes más débiles o no validar la longitud exacta del mensaje), un atacante podría interceptar una transferencia legítima (ej. "Transfer 100 euros") y modificarla en tránsito (ej. añadiendo ceros o cambiando destinatarios) logrando que una verificación mal diseñada dé el visto bueno (falsificación o modificación de datos).

c) Modifica el código para que no haya el problema identificado en b) e indica qué ocurre si el mensaje cambia, aunque sea un byte

In [2]:
# VERIFICACIÓN CORREGIDA
try:
    public_key.verify(
        signature,
        message,               # Usamos el mensaje original exacto
        padding.PKCS1v15(),
        hashes.SHA256()        # Usamos el mismo algoritmo hash (SHA256)
    )
    print("Firma válida: El mensaje es auténtico y no ha sido alterado.")
except Exception as e:
    print("Firma inválida:", e)

Firma válida: El mensaje es auténtico y no ha sido alterado.


¿Qué ocurre si el mensaje cambia aunque sea un byte?

- Debido a la propiedad matemática de las funciones hash (conocida como efecto avalancha), si modificas un solo byte del mensaje (por ejemplo, cambiar 100 por 200), el hash SHA-256 resultante cambiará por completo. Al intentar verificarlo, el hash recalculado no coincidirá con el hash obtenido al descifrar la firma con la clave pública, por lo que saltará una excepción y el programa imprimirá "Firma inválida".

d) ¿Es PKCS1v15 una buena alternativa? De no ser así ¿Qué se utiliza?

- No es la mejor alternativa para implementaciones modernas. El esquema de relleno PKCS#1 v1.5 es vulnerable a ciertos ataques avanzados (como ataques de relleno o Padding Oracle Attacks) si no se implementa con contramedidas estrictas.

- Alternativa actual: Hoy en día se recomienda encarecidamente utilizar PSS (Probabilistic Signature Scheme) (padding.PSS), que incorpora aleatoriedad (un salt) en el proceso de relleno, ofreciendo una seguridad demostrable mucho más sólida frente a ataques criptográficos.

e) ¿Podría un atacante modificar el mensaje y aun así pasar la verificación en algún escenario mal diseñado?

- Sí, podría ocurrir en escenarios con fallos graves de diseño:

    - Manejo incorrecto de excepciones: Si el código del servidor captura la excepción de fallo de firma de manera genérica y aun así permite continuar con la ejecución del programa.

    - Colisiones de Hash: Si se utilizasen funciones hash obsoletas y vulnerables a colisiones (como MD5 o SHA-1), un atacante podría calcular dos mensajes distintos que produzcan el mismo hash. (Afortunadamente, SHA-256 es seguro frente a esto hoy en día).

    - Reutilización de parámetros o claves débiles.

## Ejercicio 2: Ataques de Repetición (Replay Attacks) en Firmas Digitales

### Razonamiento Previo


- La firma digital garantiza integridad (el mensaje no fue modificado), autenticidad (lo firmó el dueño de la clave privada) y no repudio.

- Sin embargo, la firma digital por sí sola no tiene memoria ni noción del tiempo o del contexto. Si un mensaje dice únicamente "Transfer 100 euros" y lleva una firma criptográfica válida, el sistema receptor solo puede comprobar que el emisor autorizó esa orden, pero no sabe si esa orden ya fue ejecutada hace 5 minutos o hace 3 días.

a) Asumiendo que la verificación es correcta, ¿se podría hacer un replay attack?

Sí, totalmente. Un atacante que intercepte el tráfico de red (ataque Man-in-the-Middle o sniffing) captura tanto el mensaje original (b"Transfer 100 euros") como su firma digital válida (signature). Aunque el atacante no puede modificar el contenido del mensaje (porque invalidaría la firma), sí puede volver a enviar exactamente el mismo paquete al servidor bancario repetidas veces. Como el mensaje y la firma siguen coincidiendo matemáticamente, el receptor los dará por válidos cada vez y ejecutará múltiples transferencias de 100 euros.   

b) ¿Cómo se podría solucionar este problema?

Para evitar un replay attack, necesitamos que cada mensaje firmado sea único y tenga validez temporal o de un solo uso, incluyendo esta información dentro de los datos que se firman (para que el atacante no pueda alterarla). Las tres soluciones estándar son:   

1. Uso de un Nonce (Number used once): 
    - Se genera un número aleatorio único (o un identificador UUID) para cada transacción.
    - El emisor firma el conjunto: Mensaje + Nonce.
    - El servidor receptor guarda una base de datos/caché de los nonces ya utilizados. Si recibe una firma válida cuyo nonce ya fue procesado anteriormente, rechaza la petición.
    
2. Uso de Marcas de Tiempo (Timestamps):
    - Se incluye la fecha y hora exacta de creación dentro del mensaje firmado: Mensaje + Timestamp.
    - El servidor comprueba que la firma sea válida y que el timestamp esté dentro de una ventana de tiempo aceptable (por ejemplo, que no tenga más de 30 segundos de antigüedad) y sea posterior al último mensaje recibido. (Requiere que los relojes del emisor y receptor estén sincronizados mediante NTP).

3. Números de Secuencia (Contadores):
    - Ambas partes mantienen un contador sincronizado ($1, 2, 3...$). Cada mensaje incluye su número correlativo (Mensaje + N_secuencia) y el receptor solo acepta el número inmediatamente siguiente al último procesado.

In [3]:
import os
import time
from cryptography.hazmat.primitives import hashes
from cryptography.hazmat.primitives.asymmetric import padding

# Base de datos simulada del servidor para guardar nonces ya usados
used_nonces = set()

def crear_transferencia_segura(private_key, mensaje_base):
    # 1. Generamos un nonce aleatorio de 16 bytes y el timestamp actual
    nonce = os.urandom(16).hex()
    timestamp = str(int(time.time()))
    
    # 2. Construimos el payload completo que será firmado
    payload = f"{mensaje_base}|nonce:{nonce}|ts:{timestamp}".encode('utf-8')
    
    # 3. Firmamos TODO el payload (usando PSS, el estándar moderno)
    signature = private_key.sign(
        payload,
        padding.PSS(
            mgf=padding.MGF1(hashes.SHA256()),
            salt_length=padding.PSS.MAX_LENGTH
        ),
        hashes.SHA256()
    )
    return payload, signature

def procesar_transferencia(public_key, payload, signature):
    try:
        # 1. Verificar firma criptográfica
        public_key.verify(
            signature,
            payload,
            padding.PSS(
                mgf=padding.MGF1(hashes.SHA256()),
                salt_length=padding.PSS.MAX_LENGTH
            ),
            hashes.SHA256()
        )
        
        # 2. Extraer nonce y timestamp del payload
        partes = payload.decode('utf-8').split('|')
        nonce = partes[1].split(':')[1]
        timestamp = int(partes[2].split(':')[1])
        
        # 3. Comprobar ventana de tiempo (máximo 30 segundos de antigüedad)
        if abs(time.time() - timestamp) > 30:
            return "RECHAZADO: El mensaje ha expirado (Timestamp antiguo)."
            
        # 4. Comprobar si el nonce ya fue usado (Protección Anti-Replay)
        if nonce in used_nonces:
            return "RECHAZADO: ¡Replay Attack detectado! Este nonce ya fue utilizado."
            
        # Registrar nonce y aprobar operación
        used_nonces.add(nonce)
        return f"APROBADO: Ejecutando -> {partes[0]}"
        
    except Exception:
        return "RECHAZADO: Firma digital inválida."

# --- PRUEBA DE EJECUCIÓN ---
# Reutilizamos las claves (private_key, public_key) del Ejercicio 1
payload_enviado, firma_enviada = crear_transferencia_segura(private_key, "Transfer 100 euros")

print("1er envío (Legítimo):", procesar_transferencia(public_key, payload_enviado, firma_enviada))
print("2do envío (Atacante repite el paquete):", procesar_transferencia(public_key, payload_enviado, firma_enviada))

1er envío (Legítimo): APROBADO: Ejecutando -> Transfer 100 euros
2do envío (Atacante repite el paquete): RECHAZADO: ¡Replay Attack detectado! Este nonce ya fue utilizado.


## Ejercicio 3:

### Razonamiento previo

- Se invoca el método public_key.encrypt(...) junto con padding.OAEP(...) y padding.MGF1(...).   
- Según el Módulo 5 y la Práctica 3, en el criptosistema RSA la clave pública se utiliza para cifrar ($C \equiv M^e \pmod n$) o verificar firmas, mientras que la clave privada se utiliza para descifrar ($M \equiv C^d \pmod n$) o firmar.   
- Además, al emplear RSA-OAEP (Optimal Asymmetric Encryption Padding) con la función de generación de máscara MGF1, convertimos el cifrado RSA en un esquema probabilístico que inyecta aleatoriedad antes de la exponenciación modular, protegiéndolo frente a ataques de texto claro escogido o ataques deterministas.

version original

In [4]:
ciphertext = public_key.encrypt(
 message,
 padding.OAEP(
 mgf=padding.MGF1(algorithm=hashes.SHA256()),
 algorithm=hashes.SHA256(),
 label=None
 )
)

version modificada

In [5]:
# Reutilizamos el par de claves RSA (private_key, public_key) y 'message' del Ejercicio 1

# 1. CIFRADO ASIMÉTRICO (código del enunciado)
ciphertext = public_key.encrypt(
    message,
    padding.OAEP(
        mgf=padding.MGF1(algorithm=hashes.SHA256()),
        algorithm=hashes.SHA256(),
        label=None
    )
)

print("Texto cifrado (en bytes):", ciphertext[:32], "... (longitud:", len(ciphertext), "bytes)")

# 2. DESCIFRADO ASIMÉTRICO (usando la clave privada correspondiente)
plaintext = private_key.decrypt(
    ciphertext,
    padding.OAEP(
        mgf=padding.MGF1(algorithm=hashes.SHA256()),
        algorithm=hashes.SHA256(),
        label=None
    )
)

print("Mensaje descifrado:", plaintext)

Texto cifrado (en bytes): b'\x91\xc1\x96\xd0\x9c\xa7\x1b1~(\x96\xd8\x9e\xd0jRt\x14I4?\xb9`c\x90\x15\xfd\x89I#K\xe1' ... (longitud: 256 bytes)
Mensaje descifrado: b'Transfer 100 euros'


a) ¿Qué es lo que hace el código anterior?

- Ejecuta una operación de cifrado asimétrico sobre la variable message utilizando la clave pública (public_key) del destinatario mediante el método encrypt.   
- Antes de aplicar la operación matemática de RSA ($C \equiv M^e \pmod n$), aplica el esquema de relleno probabilístico seguro OAEP (Optimal Asymmetric Encryption Padding).   
- Configura dicho esquema utilizando la función de generación de máscara MGF1 basada en SHA-256 y establece SHA-256 como algoritmo hash interno del relleno (sin etiqueta asociada, label=None). El resultado almacenado en ciphertext es el bloque de bytes cifrado.   


b) ¿Qué propiedad de seguridad proporciona?

- Proporciona Confidencialidad.   
- Al cifrarse con la clave pública del receptor, únicamente el legítimo poseedor de la clave privada correspondiente (private_key) podrá ejecutar private_key.decrypt() para recuperar el texto plano original.   
- (Nota importante): Por sí solo, este código no proporciona autenticidad ni no repudio, ya que la clave pública es conocida por cualquiera y cualquier usuario podría haber cifrado un mensaje con ella.   

c) ¿En qué se diferencia del código proporcionado en 1)?

| Criterio de comparación | Código del Ejercicio 1 (`sign` / `verify`) | Código del Ejercicio 3 (`encrypt`) |
| :--- | :--- | :--- |
| **Propósito y propiedad de seguridad** | **Firma digital:** Garantiza **autenticidad, integridad y no repudio** del emisor. El mensaje viaja en claro acompañado de su firma. | **Cifrado asimétrico:** Garantiza **confidencialidad**. Oculta el contenido del mensaje transformándolo en `ciphertext`. |
| **Uso de las claves (Asimetría)** | Se firma con la **clave privada** (`private_key.sign`) y se verifica con la **clave pública** (`public_key.verify`). | Se cifra con la **clave pública** (`public_key.encrypt`) y se descifra con la **clave privada** (`private_key.decrypt`). |
| **Esquema de relleno (*Padding*)** | Usa `padding.PKCS1v15()` (tradicional y desaconsejado; el estándar moderno recomendado para firma es `padding.PSS`). | Usa `padding.OAEP` combinado con `padding.MGF1`, que es el esquema probabilístico recomendado para cifrado RSA. |
| **Tratamiento y tamaño de los datos** | `sign()` calcula primero el resumen **hash** (`SHA-256`) del mensaje y firma dicho resumen, permitiendo mensajes de cualquier longitud. | `encrypt()` cifra directamente el texto plano corto (limitado al tamaño de la clave RSA menos el relleno OAEP; para mensajes largos se recurre al **Cifrado Híbrido**). |